# 🐍 Python Decorators — Complete A–Z Learning Guide

## Project 24 — Decorators

Decorators are an advanced Python feature built on top of **functions as
objects, nested functions, closures, and higher-order functions**.

### Learning Roadmap

```text
FUNCTIONS AS OBJECTS
        ↓
NESTED FUNCTIONS
        ↓
HIGHER-ORDER FUNCTIONS
        ↓
CLOSURES
        ↓
DECORATOR
        ↓
@decorator SYNTAX
        ↓
functools.wraps
        ↓
ARGUMENTS: *args / **kwargs
        ↓
RETURN VALUES
        ↓
STACKING DECORATORS
        ↓
CLASS-BASED DECORATORS
        ↓
BUILT-IN DECORATORS
        ↓
REAL-WORLD PROJECTS
```

# 1. Functions Are Objects

In Python, functions are first-class objects.

That means a function can be:

- assigned to a variable
- passed as an argument
- returned from another function
- stored in a collection

This idea is the foundation of decorators.

In [1]:
def greet(name):
    return f"Hello, {name}"

say_hello = greet

print(say_hello("Kaif"))
print(greet is say_hello)

Hello, Kaif
True


# 2. Passing Functions as Arguments

Because functions are objects, another function can receive a function as an
argument.

This is a **higher-order function** pattern.

In [2]:
def square(number):
    return number ** 2

def apply_operation(function, value):
    return function(value)

print(apply_operation(square, 5))

25


# 3. Returning Functions

A function can create and return another function.

The returned function can remember values from the surrounding scope.

In [3]:
def create_greeting(prefix):
    def greet(name):
        return f"{prefix}, {name}!"

    return greet

hello = create_greeting("Hello")
welcome = create_greeting("Welcome")

print(hello("Kaif"))
print(welcome("Kaif"))

Hello, Kaif!
Welcome, Kaif!


# 4. Nested Functions

A function defined inside another function is called a **nested function**.

Nested functions are commonly used when helper behavior should remain local
to the outer function.

In [4]:
def calculate_total(price, tax_rate):
    def add_tax(amount):
        return amount * (1 + tax_rate)

    return add_tax(price)

print(calculate_total(1000, 0.18))

1180.0


# 5. Closures

A **closure** occurs when an inner function remembers values from its enclosing
scope even after the outer function has finished.

```text
OUTER FUNCTION
      ↓
creates value
      ↓
INNER FUNCTION
      ↓
remembers value
      ↓
returned to caller
```

In [5]:
def make_multiplier(factor):
    def multiply(number):
        return number * factor

    return multiply

double = make_multiplier(2)
triple = make_multiplier(3)

print(double(10))
print(triple(10))

20
30


# 6. Understanding Closure State

The inner function can access the enclosing variable because that value is
captured by the closure.

In [6]:
def make_counter():
    count = 0

    def increment():
        nonlocal count
        count += 1
        return count

    return increment

counter = make_counter()

print(counter())
print(counter())
print(counter())

1
2
3


# 7. What Is a Decorator?

A **decorator** is a callable that takes another callable, extends or changes
its behavior, and returns a callable.

Conceptually:

```text
ORIGINAL FUNCTION
       ↓
    DECORATOR
       ↓
WRAPPED FUNCTION
       ↓
ENHANCED BEHAVIOR
```

In [7]:
def uppercase_decorator(function):
    def wrapper():
        return function().upper()

    return wrapper

def message():
    return "hello python"

message = uppercase_decorator(message)

print(message())

HELLO PYTHON


# 8. Decorator Syntax — `@decorator`

Python provides convenient syntax for applying a decorator.

Instead of:

```python
function = decorator(function)
```

you can write:

```python
@decorator
def function():
    ...
```

In [8]:
def uppercase_decorator(function):
    def wrapper():
        return function().upper()

    return wrapper

@uppercase_decorator
def message():
    return "hello python"

print(message())

HELLO PYTHON


# 9. Decorators with Function Arguments

Real functions usually accept arguments, so wrappers commonly use:

```python
*args
**kwargs
```

This allows the decorator to work with many different function signatures.

In [9]:
def log_call(function):
    def wrapper(*args, **kwargs):
        print("Calling:", function.__name__)
        result = function(*args, **kwargs)
        print("Finished:", function.__name__)
        return result

    return wrapper

@log_call
def add(a, b):
    return a + b

print("Result:", add(10, 20))

Calling: add
Finished: add
Result: 30


# 10. Returning the Original Result

A wrapper should normally return the wrapped function's result when the
decorator is intended only to add side effects.

In [10]:
def trace(function):
    def wrapper(*args, **kwargs):
        print("START")
        result = function(*args, **kwargs)
        print("END")
        return result

    return wrapper

@trace
def multiply(a, b):
    return a * b

result = multiply(5, 4)

print("Result:", result)

START
END
Result: 20


# 11. `functools.wraps`

A wrapper can otherwise hide useful metadata such as the original function's
name and docstring.

Use:

```python
from functools import wraps
```

and:

```python
@wraps(function)
```

inside the decorator.

In [11]:
from functools import wraps

def log_call(function):
    @wraps(function)
    def wrapper(*args, **kwargs):
        return function(*args, **kwargs)

    return wrapper

@log_call
def calculate_total(a, b):
    """Calculate the total of two values."""
    return a + b

print(calculate_total.__name__)
print(calculate_total.__doc__)

calculate_total
Calculate the total of two values.


# 12. Why `wraps` Matters

Without `functools.wraps`, introspection can show the wrapper instead of the
original function.

With `wraps`, important metadata is preserved through the standard decorator
pattern.

In [12]:
from functools import wraps

def decorator(function):
    @wraps(function)
    def wrapper(*args, **kwargs):
        return function(*args, **kwargs)

    return wrapper

@decorator
def greet(name):
    """Greet a customer."""
    return f"Hello, {name}"

print(greet.__name__)
print(greet.__doc__)

greet
Greet a customer.


# 13. Timing Decorator

A decorator can measure execution time.

For benchmarking, remember that one timing measurement is not a complete
performance study.

In [13]:
from functools import wraps
from time import perf_counter

def measure_time(function):
    @wraps(function)
    def wrapper(*args, **kwargs):
        start = perf_counter()
        result = function(*args, **kwargs)
        elapsed = perf_counter() - start
        print(f"{function.__name__}: {elapsed:.6f} seconds")
        return result

    return wrapper

@measure_time
def calculate():
    return sum(range(100_000))

print("Result:", calculate())

calculate: 0.003296 seconds
Result: 4999950000


# 14. Validation Decorator

Decorators can perform reusable validation before calling a function.

In [14]:
from functools import wraps

def require_positive(function):
    @wraps(function)
    def wrapper(value, *args, **kwargs):
        if value <= 0:
            raise ValueError("Value must be positive")
        return function(value, *args, **kwargs)

    return wrapper

@require_positive
def calculate_square(value):
    return value ** 2

print(calculate_square(8))

64


# 15. Authentication-Style Decorator

A decorator can enforce a condition before a protected operation.

This example is educational; production authentication should use a proper
security architecture rather than a simple boolean.

In [15]:
from functools import wraps

def requires_login(function):
    @wraps(function)
    def wrapper(user_logged_in, *args, **kwargs):
        if not user_logged_in:
            raise PermissionError("Login required")
        return function(user_logged_in, *args, **kwargs)

    return wrapper

@requires_login
def view_dashboard(user_logged_in):
    return "Dashboard opened"

print(view_dashboard(True))

Dashboard opened


# 16. Decorator That Accepts Its Own Arguments

Sometimes the decorator itself needs configuration.

This creates three layers:

```text
decorator_factory
      ↓
decorator
      ↓
wrapper
```

Example:

```python
@repeat(3)
def greet():
    ...
```

In [16]:
from functools import wraps

def repeat(times):
    def decorator(function):
        @wraps(function)
        def wrapper(*args, **kwargs):
            result = None

            for _ in range(times):
                result = function(*args, **kwargs)

            return result

        return wrapper

    return decorator

@repeat(3)
def greet():
    print("Hello")

greet()

Hello
Hello
Hello


# 17. Decorator With `*args` and `**kwargs`

A generic decorator should normally forward positional and keyword arguments
unchanged.

In [17]:
from functools import wraps

def debug(function):
    @wraps(function)
    def wrapper(*args, **kwargs):
        print("args:", args)
        print("kwargs:", kwargs)

        result = function(*args, **kwargs)

        print("result:", result)
        return result

    return wrapper

@debug
def create_user(name, role="Analyst"):
    return {"name": name, "role": role}

print(create_user("Kaif", role="Data Scientist"))

args: ('Kaif',)
kwargs: {'role': 'Data Scientist'}
result: {'name': 'Kaif', 'role': 'Data Scientist'}
{'name': 'Kaif', 'role': 'Data Scientist'}


# 18. Decorators and Return Values

Decorators can:

1. return the original result
2. transform the result
3. replace the result
4. intentionally suppress a result

The behavior should be documented because changing a function's return value
can surprise callers.

In [18]:
from functools import wraps

def format_result(function):
    @wraps(function)
    def wrapper(*args, **kwargs):
        result = function(*args, **kwargs)
        return f"Result = {result}"

    return wrapper

@format_result
def add(a, b):
    return a + b

print(add(10, 20))

Result = 30


# 19. Decorator Order

Multiple decorators can be stacked.

```python
@decorator_a
@decorator_b
def function():
    ...
```

This is conceptually equivalent to:

```python
function = decorator_a(decorator_b(function))
```

The order matters.

In [19]:
def first(function):
    @wraps(function)
    def wrapper(*args, **kwargs):
        print("First before")
        result = function(*args, **kwargs)
        print("First after")
        return result
    return wrapper

def second(function):
    @wraps(function)
    def wrapper(*args, **kwargs):
        print("Second before")
        result = function(*args, **kwargs)
        print("Second after")
        return result
    return wrapper

@first
@second
def greet():
    print("Hello")

greet()

First before
Second before
Hello
Second after
First after


# 20. Built-in Decorator — `@staticmethod`

`@staticmethod` defines a method that does not receive an automatic `self`
instance argument.

It is useful when a function belongs logically to a class but does not need
instance state.

In [20]:
class Temperature:
    @staticmethod
    def celsius_to_fahrenheit(celsius):
        return (celsius * 9 / 5) + 32

print(Temperature.celsius_to_fahrenheit(25))

77.0


# 21. Built-in Decorator — `@classmethod`

`@classmethod` creates a method that receives the class as `cls`.

It is often used for alternative constructors or class-level behavior.

In [21]:
class Employee:
    company = "DataTech"

    def __init__(self, name):
        self.name = name

    @classmethod
    def from_name(cls, name):
        return cls(name)

employee = Employee.from_name("Kaif")

print(employee.name)
print(employee.company)

Kaif
DataTech


# 22. Built-in Decorator — `@property`

`@property` allows method-based logic to be accessed with attribute syntax.

In [22]:
class Product:
    def __init__(self, price, tax_rate):
        self.price = price
        self.tax_rate = tax_rate

    @property
    def final_price(self):
        return self.price * (1 + self.tax_rate)

product = Product(1000, 0.18)

print(product.final_price)

1180.0


# 23. Property Setter and Decorator Syntax

A property can provide controlled assignment with:

```python
@property
def value(self):
    ...

@value.setter
def value(self, new_value):
    ...
```

In [23]:
class Employee:
    def __init__(self, salary):
        self.salary = salary

    @property
    def salary(self):
        return self._salary

    @salary.setter
    def salary(self, value):
        if value < 0:
            raise ValueError("Salary cannot be negative")
        self._salary = value

employee = Employee(50000)
employee.salary = 60000

print(employee.salary)

60000


# 24. Decorator Factory Pattern

A decorator factory returns a decorator.

This pattern is useful when configuration is needed.

In [24]:
from functools import wraps

def add_prefix(prefix):
    def decorator(function):
        @wraps(function)
        def wrapper(*args, **kwargs):
            result = function(*args, **kwargs)
            return f"{prefix}{result}"
        return wrapper
    return decorator

@add_prefix("[INFO] ")
def message():
    return "Application started"

print(message())

[INFO] Application started


# 25. Class-Based Decorator

A class can also act as a decorator by implementing `__call__()`.

Instances of the class become callable.

In [25]:
from functools import update_wrapper

class CallCounter:
    def __init__(self, function):
        self.count = 0
        update_wrapper(self, function)
        self.function = function

    def __call__(self, *args, **kwargs):
        self.count += 1
        return self.function(*args, **kwargs)

@CallCounter
def add(a, b):
    return a + b

print(add(2, 3))
print(add(4, 5))
print("Calls:", add.count)

5
9
Calls: 2


# 26. Decorators and Exceptions

A decorator can log or handle exceptions, but it should not silently hide
failures unless that is the explicit design.

In [26]:
from functools import wraps

def log_errors(function):
    @wraps(function)
    def wrapper(*args, **kwargs):
        try:
            return function(*args, **kwargs)
        except Exception as error:
            print(f"{function.__name__} failed: {error}")
            raise

    return wrapper

@log_errors
def divide(a, b):
    return a / b

try:
    divide(10, 0)
except ZeroDivisionError:
    print("Handled by caller")

divide failed: division by zero
Handled by caller


# 27. Decorators for Caching

Python's standard library provides `functools.lru_cache` for memoization.

For pure or appropriately cacheable functions, repeated calls can reuse
previous results.

In [27]:
from functools import lru_cache

@lru_cache(maxsize=None)
def fibonacci(number):
    if number < 2:
        return number

    return fibonacci(number - 1) + fibonacci(number - 2)

print(fibonacci(20))
print(fibonacci.cache_info())

6765
CacheInfo(hits=18, misses=21, maxsize=None, currsize=21)


# 28. Decorator for Retry Logic

A retry decorator can repeat an operation after selected failures.

In production systems, retry policies should normally include limits,
appropriate exception selection, and often backoff.

In [28]:
from functools import wraps

def retry(times):
    def decorator(function):
        @wraps(function)
        def wrapper(*args, **kwargs):
            last_error = None

            for _ in range(times):
                try:
                    return function(*args, **kwargs)
                except ValueError as error:
                    last_error = error

            raise last_error

        return wrapper

    return decorator

attempts = {"count": 0}

@retry(3)
def unstable_operation():
    attempts["count"] += 1

    if attempts["count"] < 3:
        raise ValueError("Temporary failure")

    return "Success"

print(unstable_operation())

Success


# 29. Common Mistakes

### Mistake 1 — Forgetting to return the wrapper

```python
def decorator(function):
    def wrapper():
        ...
```

The decorator must return the callable that should replace the original.

### Mistake 2 — Forgetting `*args` / `**kwargs`

The decorator may then work only for one specific function signature.

### Mistake 3 — Forgetting `@wraps`

Function metadata can become misleading.

### Mistake 4 — Changing return values accidentally

A wrapper that does not return the wrapped result can change the API.

### Mistake 5 — Hiding exceptions

Silently swallowing errors can make debugging difficult.

### Mistake 6 — Stacking decorators without understanding order

Decorator order can change behavior.

### Mistake 7 — Overusing decorators

If a normal function is clearer, use the simpler design.

# 30. Best Practices

- Keep decorators small and focused.
- Use `functools.wraps` for function-based decorators.
- Forward `*args` and `**kwargs` when appropriate.
- Preserve the wrapped function's return value unless intentionally changing it.
- Do not silently swallow exceptions.
- Document decorators that alter behavior.
- Keep decorator order intentional.
- Avoid hidden side effects.
- Use standard-library decorators when they solve the problem well.
- Prefer simple code when a decorator adds unnecessary abstraction.

# 🚀 Project 1 — 01 — Function Logger

## Business Problem

Log which business function is being called and preserve its metadata.

## Architecture

```text
INPUT
 ↓
DECORATOR
 ↓
VALIDATE / LOG / TIME / CONTROL
 ↓
ORIGINAL FUNCTION
 ↓
RESULT
```

## Implementation

In [29]:
from functools import wraps

def log_call(function):
    @wraps(function)
    def wrapper(*args, **kwargs):
        print(f"Calling {function.__name__}")
        result = function(*args, **kwargs)
        print(f"Finished {function.__name__}")
        return result
    return wrapper

@log_call
def calculate_sales(a, b):
    return a + b

print("Sales:", calculate_sales(1000, 2500))

Calling calculate_sales
Finished calculate_sales
Sales: 3500


# 🚀 Project 2 — 02 — Execution Timer

## Business Problem

Measure execution time for selected functions.

## Architecture

```text
INPUT
 ↓
DECORATOR
 ↓
VALIDATE / LOG / TIME / CONTROL
 ↓
ORIGINAL FUNCTION
 ↓
RESULT
```

## Implementation

In [30]:
from functools import wraps
from time import perf_counter

def timer(function):
    @wraps(function)
    def wrapper(*args, **kwargs):
        start = perf_counter()
        result = function(*args, **kwargs)
        elapsed = perf_counter() - start
        print(f"{function.__name__}: {elapsed:.6f}s")
        return result
    return wrapper

@timer
def process_data():
    return sum(number * number for number in range(100_000))

print("Result:", process_data())

process_data: 0.018272s
Result: 333328333350000


# 🚀 Project 3 — 03 — Input Validation

## Business Problem

Create a reusable decorator that validates a numeric argument.

## Architecture

```text
INPUT
 ↓
DECORATOR
 ↓
VALIDATE / LOG / TIME / CONTROL
 ↓
ORIGINAL FUNCTION
 ↓
RESULT
```

## Implementation

In [31]:
from functools import wraps

def positive_only(function):
    @wraps(function)
    def wrapper(value, *args, **kwargs):
        if value <= 0:
            raise ValueError("Value must be positive")
        return function(value, *args, **kwargs)
    return wrapper

@positive_only
def calculate_discounted_price(price, rate):
    return price * (1 - rate)

print(calculate_discounted_price(1000, 0.10))

900.0


# 🚀 Project 4 — 04 — Permission Checker

## Business Problem

Protect an operation using a reusable permission decorator.

## Architecture

```text
INPUT
 ↓
DECORATOR
 ↓
VALIDATE / LOG / TIME / CONTROL
 ↓
ORIGINAL FUNCTION
 ↓
RESULT
```

## Implementation

In [32]:
from functools import wraps

def requires_role(required_role):
    def decorator(function):
        @wraps(function)
        def wrapper(user_role, *args, **kwargs):
            if user_role != required_role:
                raise PermissionError(
                    f"Required role: {required_role}"
                )
            return function(user_role, *args, **kwargs)
        return wrapper
    return decorator

@requires_role("admin")
def delete_report(user_role, report_id):
    return f"Report {report_id} deleted"

print(delete_report("admin", 101))

Report 101 deleted


# 🚀 Project 5 — 05 — Retry Decorator

## Business Problem

Retry a selected operation a limited number of times.

## Architecture

```text
INPUT
 ↓
DECORATOR
 ↓
VALIDATE / LOG / TIME / CONTROL
 ↓
ORIGINAL FUNCTION
 ↓
RESULT
```

## Implementation

In [33]:
from functools import wraps

def retry(times):
    def decorator(function):
        @wraps(function)
        def wrapper(*args, **kwargs):
            last_error = None

            for _ in range(times):
                try:
                    return function(*args, **kwargs)
                except ValueError as error:
                    last_error = error

            raise last_error
        return wrapper
    return decorator

state = {"attempts": 0}

@retry(3)
def connect():
    state["attempts"] += 1

    if state["attempts"] < 3:
        raise ValueError("Temporary connection failure")

    return "Connected"

print(connect())

Connected


# 🚀 Project 6 — 06 — Result Formatter

## Business Problem

Standardize the display of business function results.

## Architecture

```text
INPUT
 ↓
DECORATOR
 ↓
VALIDATE / LOG / TIME / CONTROL
 ↓
ORIGINAL FUNCTION
 ↓
RESULT
```

## Implementation

In [34]:
from functools import wraps

def report_result(function):
    @wraps(function)
    def wrapper(*args, **kwargs):
        result = function(*args, **kwargs)
        return {
            "function": function.__name__,
            "result": result,
        }
    return wrapper

@report_result
def calculate_revenue(sales):
    return sum(sales)

print(calculate_revenue([1000, 2500, 1800]))

{'function': 'calculate_revenue', 'result': 5300}


# 🚀 Project 7 — 07 — Audit Decorator

## Business Problem

Create a simple audit trail around important operations.

## Architecture

```text
INPUT
 ↓
DECORATOR
 ↓
VALIDATE / LOG / TIME / CONTROL
 ↓
ORIGINAL FUNCTION
 ↓
RESULT
```

## Implementation

In [35]:
from functools import wraps
from datetime import datetime

audit_log = []

def audit(action):
    def decorator(function):
        @wraps(function)
        def wrapper(*args, **kwargs):
            result = function(*args, **kwargs)

            audit_log.append({
                "action": action,
                "function": function.__name__,
                "time": datetime.now().isoformat(timespec="seconds"),
            })

            return result
        return wrapper
    return decorator

@audit("CREATE_REPORT")
def create_report(name):
    return f"Report '{name}' created"

print(create_report("Sales"))
print(audit_log)

Report 'Sales' created
[{'action': 'CREATE_REPORT', 'function': 'create_report', 'time': '2026-09-01T19:37:00'}]


# 🚀 Project 8 — 08 — Cache Expensive Calculation

## Business Problem

Use a standard-library caching decorator for repeated calculations.

## Architecture

```text
INPUT
 ↓
DECORATOR
 ↓
VALIDATE / LOG / TIME / CONTROL
 ↓
ORIGINAL FUNCTION
 ↓
RESULT
```

## Implementation

In [36]:
from functools import lru_cache

@lru_cache(maxsize=128)
def calculate_score(customer_id):
    print("Calculating:", customer_id)
    return customer_id * 100

print(calculate_score(10))
print(calculate_score(10))
print(calculate_score.cache_info())

Calculating: 10
1000
1000
CacheInfo(hits=1, misses=1, maxsize=128, currsize=1)


# 🚀 Project 9 — 09 — Class-Based Call Counter

## Business Problem

Use a callable class as a decorator to maintain state between calls.

## Architecture

```text
INPUT
 ↓
DECORATOR
 ↓
VALIDATE / LOG / TIME / CONTROL
 ↓
ORIGINAL FUNCTION
 ↓
RESULT
```

## Implementation

In [37]:
from functools import update_wrapper

class CallCounter:
    def __init__(self, function):
        self.function = function
        self.count = 0
        update_wrapper(self, function)

    def __call__(self, *args, **kwargs):
        self.count += 1
        return self.function(*args, **kwargs)

@CallCounter
def calculate_total(a, b):
    return a + b

print(calculate_total(10, 20))
print(calculate_total(30, 40))
print("Calls:", calculate_total.count)

30
70
Calls: 2


# 🚀 Project 10 — 10 — Analytics Function Pipeline

## Business Problem

Combine decorators for logging, timing, validation, and a business calculation.

## Architecture

```text
INPUT
 ↓
DECORATOR
 ↓
VALIDATE / LOG / TIME / CONTROL
 ↓
ORIGINAL FUNCTION
 ↓
RESULT
```

## Implementation

In [38]:
from functools import wraps
from time import perf_counter

def log_call(function):
    @wraps(function)
    def wrapper(*args, **kwargs):
        print("LOG:", function.__name__)
        return function(*args, **kwargs)
    return wrapper

def timer(function):
    @wraps(function)
    def wrapper(*args, **kwargs):
        start = perf_counter()
        result = function(*args, **kwargs)
        print("TIME:", f"{perf_counter() - start:.6f}s")
        return result
    return wrapper

def positive_values(function):
    @wraps(function)
    def wrapper(values, *args, **kwargs):
        if any(value < 0 for value in values):
            raise ValueError("Values must not be negative")
        return function(values, *args, **kwargs)
    return wrapper

@log_call
@timer
@positive_values
def total_sales(values):
    return sum(values)

print("Total:", total_sales([1200, 2500, 1800, 3200]))

LOG: total_sales
TIME: 0.000006s
Total: 8700


# 🧪 Practice — Beginner → Advanced

## Beginner

1. Assign a function to another variable.
2. Pass a function to another function.
3. Return a function from another function.
4. Create a nested function.
5. Build a simple closure.
6. Explain what a decorator does.
7. Write a basic decorator.
8. Use `@decorator`.

## Intermediate

9. Write a decorator using `*args` and `**kwargs`.
10. Preserve metadata using `functools.wraps`.
11. Create a logging decorator.
12. Create a validation decorator.
13. Create a timing decorator.
14. Create a decorator factory.
15. Stack two decorators.
16. Explain decorator order.

## Advanced

17. Build a class-based decorator.
18. Build a retry decorator.
19. Build a permission decorator.
20. Build an audit decorator.
21. Use `lru_cache`.
22. Create a decorator that transforms results.
23. Explain closures inside decorator factories.
24. Build a multi-decorator analytics pipeline.

# 🎤 Interview Questions

1. What is a decorator?
2. Why are functions called first-class objects in Python?
3. What is a higher-order function?
4. What is a nested function?
5. What is a closure?
6. How does a closure relate to decorators?
7. What does `@decorator` syntax mean?
8. What is the equivalent assignment for `@decorator`?
9. Why are `*args` and `**kwargs` commonly used in decorators?
10. Why should `functools.wraps` be used?
11. What happens if `wraps` is omitted?
12. How do decorator factories work?
13. What is the order of stacked decorators?
14. Difference between `@staticmethod` and a custom decorator?
15. What does `@classmethod` do?
16. What does `@property` do?
17. Can a class be used as a decorator?
18. What does `__call__()` do?
19. How can decorators handle exceptions?
20. What is memoization?
21. What does `functools.lru_cache` provide?
22. What are common risks of overusing decorators?
23. How do decorators affect function metadata?
24. How would you write a retry decorator?
25. How would you test a decorator?

# 📚 Decorator Quick Reference

| Concept | Purpose |
|---|---|
| Function as object | Functions can be passed, stored, and returned |
| Higher-order function | Function accepts/returns functions |
| Nested function | Function defined inside another function |
| Closure | Inner function remembers enclosing values |
| Decorator | Wraps/enhances another callable |
| `@decorator` | Decorator application syntax |
| `@wraps(function)` | Preserves wrapped function metadata |
| `*args` | Forward positional arguments |
| `**kwargs` | Forward keyword arguments |
| `@staticmethod` | Class method without automatic `self`/`cls` |
| `@classmethod` | Class method receiving `cls` |
| `@property` | Attribute-style managed access |
| `__call__()` | Makes an instance callable |
| `lru_cache` | Memoization/cache decorator |

## Core Pattern

```python
from functools import wraps

def my_decorator(function):
    @wraps(function)
    def wrapper(*args, **kwargs):
        # before
        result = function(*args, **kwargs)
        # after
        return result

    return wrapper

@my_decorator
def my_function(value):
    return value
```

## Decorator Factory

```python
def configure(option):
    def decorator(function):
        @wraps(function)
        def wrapper(*args, **kwargs):
            return function(*args, **kwargs)

        return wrapper

    return decorator
```

## Stacking

```python
@decorator_a
@decorator_b
def function():
    ...
```

means:

```python
function = decorator_a(decorator_b(function))
```

# 🎯 Final Master Roadmap

```text
FUNCTIONS AS OBJECTS
 ↓
HIGHER-ORDER FUNCTIONS
 ↓
NESTED FUNCTIONS
 ↓
CLOSURES
 ↓
BASIC DECORATORS
 ↓
@decorator
 ↓
*args / **kwargs
 ↓
functools.wraps
 ↓
DECORATOR FACTORIES
 ↓
STACKED DECORATORS
 ↓
staticmethod / classmethod / property
 ↓
CLASS-BASED DECORATORS
 ↓
CACHING
 ↓
RETRY / VALIDATION / LOGGING
 ↓
10 REAL-WORLD PROJECTS
 ↓
INTERVIEW READY
```

## Projects Completed

1. 📝 Function Logger
2. ⏱️ Execution Timer
3. ✅ Input Validation
4. 🔐 Permission Checker
5. 🔁 Retry Decorator
6. 📊 Result Formatter
7. 📋 Audit Decorator
8. ⚡ Cache Expensive Calculation
9. 🔢 Class-Based Call Counter
10. 🚀 Analytics Function Pipeline

# 🎯 The END → Next Journey Begins

**Thank you for following this Jupyter Notebook.**

Keep learning, keep practicing, and keep building real-world projects.

> **Learn → Practice → Analyze → Build → Improve → Grow**

See you in the next notebook. 🚀

### Until then, keep coding and keep learning! 💻🐍

**— S Mohammed Kaif**

---

<div align="center">

## 👨‍💻 S Mohammed Kaif

**Data Science • Data Analytics • Machine Learning • AI • Python**

<a href="https://github.com/Shaik-Mohammed-Kaif" target="_blank">
GitHub — S Mohammed Kaif
</a>

&nbsp;&nbsp;&nbsp;

<a href="https://www.linkedin.com/in/s-mohammed-kaif-2a500a341/" target="_blank">
LinkedIn — S Mohammed Kaif
</a>

<br><br>

**© 2026 S Mohammed Kaif**

</div>